# Mercimek Olgunluk Tespiti — YOLO11m Eğitimi

Bu defter Roboflow'da etiketlediğin tohum setiyle YOLO11m modelini eğitir ve sonunda `best.pt` dosyasını indirir.

## Çalıştırmadan önce

**1. Colab'da GPU'yu aç:** `Runtime` → `Change runtime type` → `Hardware accelerator: T4 GPU`

**2. Roboflow'da versiyon üretirken şunlara dikkat et — bu adım kritik:**

| Ayar | Değer | Neden |
|---|---|---|
| Preprocessing → Resize | **1024×1024** veya kapalı | Roboflow varsayılanı 640'tır. 640'a düşürürsen baklalar tekrar birkaç piksele iner ve tüm emek boşa gider. |
| Preprocessing → Auto-Orient | Açık | Telefon fotoğraflarındaki EXIF döndürmesini düzeltir. |
| Augmentation | **Kapalı** | Ultralytics eğitim sırasında zaten canlı augmentasyon yapıyor. Roboflow'da ayrıca üretirsen doğrulama setini şişirir. |

**3. Train/valid/test dağılımını değiştirme.** Parçalar yüklenirken aynı kaynak fotoğraftan gelenler aynı bölmeye düşecek şekilde ayrıldı. Roboflow'da "Rebalance" dersen bu korunmaz ve test skorun gerçekte olduğundan yüksek çıkar.

## Sınıflar

- `olgunlasmis` — sarı, krem, kahverengi, kurumuş bakla
- `olgunlasmamis` — yeşil bakla

In [ ]:
#@title 1. GPU kontrolu { display-mode: "form" }
# Ciktida "Tesla T4" gibi bir kart gormuyorsan Runtime > Change runtime type > T4 GPU sec.
!nvidia-smi

In [ ]:
#@title 2. Kurulum { display-mode: "form" }
%pip install -q ultralytics roboflow

import ultralytics
ultralytics.checks()

In [ ]:
#@title 3. Veri setini Roboflow'dan indir { display-mode: "form" }
from getpass import getpass
from pathlib import Path

from roboflow import Roboflow

WORKSPACE = "yincir"              #@param {type:"string"}
PROJECT   = "mercimek-olgunluk"   #@param {type:"string"}
VERSION   = 1                     #@param {type:"integer"}

# Anahtar deftere yazilmaz, her calistirmada elle girilir.
api_key = getpass("Roboflow API key: ")

project = Roboflow(api_key=api_key).workspace(WORKSPACE).project(PROJECT)
dataset = project.version(VERSION).download("yolov11")

data_yaml = Path(dataset.location) / "data.yaml"
print("\nVeri seti:", dataset.location)

In [ ]:
#@title 4. Veri setini dogrula { display-mode: "form" }
# Egitime baslamadan once kutu sayilarina bak. Bir sinifta digerinin
# onda biri kadar kutu varsa model o sinifi ogrenemez; once etiket ekle.
from collections import Counter

import yaml
from PIL import Image

cfg = yaml.safe_load(data_yaml.read_text())
names = cfg["names"] if isinstance(cfg["names"], list) else [cfg["names"][i] for i in sorted(cfg["names"])]
print("Siniflar:", names, "\n")

sizes = set()
for split in ("train", "valid", "test"):
    images = Path(dataset.location) / split / "images"
    labels = Path(dataset.location) / split / "labels"
    if not images.exists():
        continue

    per_class = Counter()
    empty = 0
    for txt in labels.glob("*.txt"):
        rows = [r for r in txt.read_text().splitlines() if r.strip()]
        if not rows:
            empty += 1
        for row in rows:
            per_class[names[int(row.split()[0])]] += 1

    photos = sorted(images.glob("*.jpg"))
    for sample in photos[:5]:
        sizes.add(Image.open(sample).size)

    counts = "  ".join(f"{k}={v}" for k, v in sorted(per_class.items()))
    print(f"{split:6s} {len(photos):4d} gorsel | {empty:3d} bos | {sum(per_class.values()):6d} kutu | {counts}")

print("\nGorsel boyutlari:", sizes)
if any(max(s) < 1024 for s in sizes):
    print("!! UYARI: Gorseller 1024'ten kucuk. Roboflow versiyonunda resize 640'ta kalmis olabilir.")
    print("   Versiyonu 1024 resize ile yeniden uretmen gerekiyor, yoksa baklalar cok kucuk kalir.")

In [ ]:
#@title 5. Egitim { display-mode: "form" }
# imgsz=1024 bu projenin en onemli ayari: parcalar 1024 kesildi, kucultursen
# baklalar birkac piksele iner. T4'te bellek yetmezse once batch'i 4'e dusur,
# imgsz'ye dokunma.
from ultralytics import YOLO

EPOCHS = 150   #@param {type:"integer"}
BATCH  = 8     #@param {type:"integer"}
IMGSZ  = 1024  #@param {type:"integer"}

model = YOLO("yolo11m.pt")

results = model.train(
    data=str(data_yaml),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    patience=30,        # 30 epoch boyunca iyilesme yoksa erken dur
    close_mosaic=15,    # son 15 epoch'ta mozaigi kapat, kutular netlessin
    cos_lr=True,
    device=0,
    workers=2,          # Colab'in CPU'su zayif, fazlasi yavaslatiyor
    amp=True,
    seed=0,
    plots=True,
    project="runs",
    name="mercimek_yolo11m",
)

run_dir = Path(model.trainer.save_dir)
print("\nCikti klasoru:", run_dir)

### Colab bağlantısı koparsa

Eğitim yarıda kalırsa baştan başlama. Aşağıdaki hücre son kaydedilen noktadan devam eder:

```python
model = YOLO(str(run_dir / "weights" / "last.pt"))
model.train(resume=True)
```

Colab ücretsiz sürümde uzun süre işlem yapmazsan oturumu düşürür. Eğitim sırasında sekmeyi açık tut.

In [ ]:
#@title 6. Test setinde olc { display-mode: "form" }
# Test seti egitimde hic gorulmedi, gercek basari burasi.
# mAP50 kabaca: 0.5+ kullanilabilir, 0.7+ iyi, 0.3 altiysa daha cok etiket lazim.
best = YOLO(str(run_dir / "weights" / "best.pt"))
metrics = best.val(data=str(data_yaml), split="test", imgsz=IMGSZ)

print(f"\nmAP50    : {metrics.box.map50:.3f}")
print(f"mAP50-95 : {metrics.box.map:.3f}\n")
for i, name in enumerate(names):
    print(f"  {name:18s} mAP50={metrics.box.ap50[i]:.3f}  P={metrics.box.p[i]:.3f}  R={metrics.box.r[i]:.3f}")

In [ ]:
#@title 7. Egitim grafiklerine bak { display-mode: "form" }
# results.png    : kayip egrileri dusuyor mu, mAP yukseliyor mu
# confusion_...  : iki sinif birbirine karisiyor mu
# val_batch..    : solda gercek etiket, sagda modelin tahmini
from IPython.display import Image as ShowImage, display

for chart in ("results.png", "confusion_matrix_normalized.png",
              "val_batch0_labels.jpg", "val_batch0_pred.jpg"):
    path = run_dir / chart
    if path.exists():
        print(chart)
        display(ShowImage(filename=str(path), width=900))

In [ ]:
#@title 8. best.pt dosyasini indir { display-mode: "form" }
# Bu dosyayi bilgisayarina indir. Icinde sinif isimleri gomulu geldigi icin
# RAZOR Auto Labeler'a yukledigin an listede olgunlasmis / olgunlasmamis gorunur.
import shutil

from google.colab import files

weights = run_dir / "weights" / "best.pt"
target = Path("/content/mercimek_yolo11m_best.pt")
shutil.copy(weights, target)

print(f"{target}  ({target.stat().st_size / 1e6:.1f} MB)")
print("Sinif isimleri:", YOLO(str(target)).names)

files.download(str(target))